# Descripción y calidad del dataset — GH-AW Workflows Dataset

Este notebook describe el dataset generado por [Miner](../README.md) a
partir de repositorios de GitHub que usan
[GitHub Agentic Workflows (GH-AW)](https://github.github.com/gh-aw/), y
revisa su calidad antes de pasar a la exploración (Notebook 2).

**Dataset publicado en Hugging Face:**
[EstebanCQ/gh-aw-workflows-dataset](https://huggingface.co/datasets/EstebanCQ/gh-aw-workflows-dataset)


## 1. Origen y carga de los datos

El dataset fue generado con el comando `miner extract` del proyecto Miner,
a partir de repositorios de GitHub identificados como usuarios de GH-AW
(`miner mine`). Está compuesto por 4 tablas Parquet relacionadas,
publicadas en Hugging Face en el enlace de arriba y disponibles localmente
en `data/raw/` para este análisis:

| Tabla | Qué representa una fila |
|---|---|
| `repositories` | Un repositorio de GitHub del que se extrajo al menos un archivo de workflow GH-AW. |
| `workflow_files` | Un archivo `.md` de workflow GH-AW (fuente), con su frontmatter completo (como JSON) y su body Markdown ya separados. |
| `workflow_locks` | El archivo `.lock.yml`/`.lock.yaml` compilado a partir del `.md` correspondiente (relación 1:1 con `workflow_files`), con su YAML completo como texto plano. |
| `frontmatter_attributes` | Un atributo individual del frontmatter de un archivo `.md`, aplanado a un par clave-valor (formato EAV: entity-attribute-value). El frontmatter de GH-AW no tiene un esquema fijo entre workflows, así que en vez de columnas rígidas se representa como filas independientes. |

Ver el esquema entidad-relación completo en
[`docs/er-diagram.md`](../docs/er-diagram.md) y el diccionario de datos en
[`docs/data-dictionary.md`](../docs/data-dictionary.md) del proyecto Miner.


In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data/raw")

repositories = pd.read_parquet(DATA_DIR / "repositories.parquet")
workflow_files = pd.read_parquet(DATA_DIR / "workflow_files.parquet")
workflow_locks = pd.read_parquet(DATA_DIR / "workflow_locks.parquet")
frontmatter_attributes = pd.read_parquet(DATA_DIR / "frontmatter_attributes.parquet")

print(f"repositories:           {repositories.shape[0]:>6} filas x {repositories.shape[1]} columnas")
print(f"workflow_files:         {workflow_files.shape[0]:>6} filas x {workflow_files.shape[1]} columnas")
print(f"workflow_locks:         {workflow_locks.shape[0]:>6} filas x {workflow_locks.shape[1]} columnas")
print(f"frontmatter_attributes: {frontmatter_attributes.shape[0]:>6} filas x {frontmatter_attributes.shape[1]} columnas")

repositories:              338 filas x 4 columnas
workflow_files:           1390 filas x 7 columnas
workflow_locks:           1390 filas x 6 columnas
frontmatter_attributes:  58281 filas x 5 columnas


## 2. Descripción de las tablas y sus relaciones

### 2.1 Estructura de cada tabla

Claves primarias (PK) y foráneas (FK) según el esquema:

- **`repositories`**: PK `repo_id`.
- **`workflow_files`**: PK `file_id`; FK `repo_id` → `repositories.repo_id`.
- **`workflow_locks`**: PK `lock_id`; FK `file_id` → `workflow_files.file_id` (relación 1:1: cada `.md` tiene exactamente un `.lock`).
- **`frontmatter_attributes`**: PK `attribute_id`; FK `file_id` → `workflow_files.file_id`.

Relación: `repositories (1) —< (N) workflow_files (1) —< (N) frontmatter_attributes`,
y `workflow_files (1) —— (1) workflow_locks`.


In [2]:
def describe_table(df: pd.DataFrame, name: str) -> pd.DataFrame:
    return pd.DataFrame({
        "columna": df.columns,
        "dtype": [str(t) for t in df.dtypes],
        "n_nulos": df.isna().sum().values,
        "pct_nulos": (df.isna().mean() * 100).round(2).values,
    }).assign(tabla=name)

resumen = pd.concat([
    describe_table(repositories, "repositories"),
    describe_table(workflow_files, "workflow_files"),
    describe_table(workflow_locks, "workflow_locks"),
    describe_table(frontmatter_attributes, "frontmatter_attributes"),
], ignore_index=True)[["tabla", "columna", "dtype", "n_nulos", "pct_nulos"]]

resumen

,tabla,columna,dtype,n_nulos,pct_nulos
0,repositories,repo_id,int64,0,0.0
1,repositories,owner,str,0,0.0
2,repositories,name,str,0,0.0
3,repositories,full_name,str,0,0.0
4,workflow_files,file_id,int64,0,0.0
5,workflow_files,repo_id,int64,0,0.0
6,workflow_files,file_name,str,0,0.0
7,workflow_files,file_path,str,0,0.0
8,workflow_files,raw_frontmatter,str,0,0.0
9,workflow_files,body_markdown,str,0,0.0


### 2.2 Repositorios y archivos únicos (distinguiendo de la cantidad de filas)

`repo_id`, `file_id` y `lock_id` son las claves primarias de `repositories`,
`workflow_files` y `workflow_locks`, así que en esas tablas cada fila ya es
una entidad única por definición. Lo que vale la pena verificar es cuántos
repositorios y archivos *distintos* participan en la tabla larga
`frontmatter_attributes`, donde el número de filas (un atributo) es mucho
mayor que el número de entidades que describe -- y que la relación 1:1
entre `workflow_files` y `workflow_locks` efectivamente se cumple en los
datos, no solo por diseño del extractor.


In [3]:
n_repos = repositories["repo_id"].nunique()
n_files = workflow_files["file_id"].nunique()
n_locks = workflow_locks["lock_id"].nunique()
n_files_con_atributos = frontmatter_attributes["file_id"].nunique()
n_repos_con_archivos = workflow_files["repo_id"].nunique()

print(f"Repositorios únicos (repositories):                  {n_repos}")
print(f"Archivos .md únicos (workflow_files):                {n_files}")
print(f"Archivos .lock únicos (workflow_locks):              {n_locks}")
print(f"Repositorios distintos con al menos 1 archivo:       {n_repos_con_archivos}")
print(f"Archivos distintos con al menos 1 atributo frontmatter: {n_files_con_atributos}")
print(f"Filas en frontmatter_attributes (NO son entidades, son atributos): {len(frontmatter_attributes)}")
print(f"Promedio de atributos de frontmatter por archivo:    {len(frontmatter_attributes) / n_files_con_atributos:.1f}")
print(f"Promedio de archivos .md por repositorio:            {n_files / n_repos:.1f}")

mismo_conjunto_ids = set(workflow_files["file_id"]) == set(workflow_locks["file_id"])
print(f"¿Cada .md tiene exactamente 1 .lock? {n_files == n_locks and mismo_conjunto_ids}")

Repositorios únicos (repositories):                  338
Archivos .md únicos (workflow_files):                1390
Archivos .lock únicos (workflow_locks):              1390
Repositorios distintos con al menos 1 archivo:       338
Archivos distintos con al menos 1 atributo frontmatter: 1389
Filas en frontmatter_attributes (NO son entidades, son atributos): 58281
Promedio de atributos de frontmatter por archivo:    42.0
Promedio de archivos .md por repositorio:            4.1
¿Cada .md tiene exactamente 1 .lock? True


## 3. Revisión de calidad

Se revisan, para las 4 tablas: valores ausentes por columna, filas
duplicadas, claves primarias repetidas o vacías, claves foráneas sin
correspondencia, consistencia interna de tipos de dato, y el cumplimiento
de la relación 1:1 entre `workflow_files` y `workflow_locks`. Importante:
la **ausencia de un campo opcional del frontmatter no se considera un
error** —`frontmatter_attributes` es una tabla larga (EAV) donde cada
archivo solo aporta las filas de los campos que efectivamente declaró; que
un workflow no use, por ejemplo, `tools`, es un dato válido, no un
problema de calidad.


In [4]:
checks = []

# --- valores ausentes por columna (ya calculado en 2.1, se resume acá) ---
total_nulos = int(resumen["n_nulos"].sum())
checks.append(("Valores ausentes (todas las columnas, las 4 tablas)", total_nulos))

# --- filas completamente duplicadas ---
checks.append(("Filas duplicadas en repositories", int(repositories.duplicated().sum())))
checks.append(("Filas duplicadas en workflow_files", int(workflow_files.duplicated().sum())))
checks.append(("Filas duplicadas en workflow_locks", int(workflow_locks.duplicated().sum())))
checks.append(("Filas duplicadas en frontmatter_attributes", int(frontmatter_attributes.duplicated().sum())))

# --- claves primarias vacías o repetidas ---
checks.append(("repo_id vacío o duplicado", int(repositories["repo_id"].isna().sum() + repositories["repo_id"].duplicated().sum())))
checks.append(("file_id vacío o duplicado", int(workflow_files["file_id"].isna().sum() + workflow_files["file_id"].duplicated().sum())))
checks.append(("lock_id vacío o duplicado", int(workflow_locks["lock_id"].isna().sum() + workflow_locks["lock_id"].duplicated().sum())))
checks.append(("attribute_id vacío o duplicado", int(frontmatter_attributes["attribute_id"].isna().sum() + frontmatter_attributes["attribute_id"].duplicated().sum())))

# --- claves foráneas sin correspondencia ---
fk_files_huerfanas = int((~workflow_files["repo_id"].isin(repositories["repo_id"])).sum())
fk_locks_huerfanas = int((~workflow_locks["file_id"].isin(workflow_files["file_id"])).sum())
fk_attrs_huerfanas = int((~frontmatter_attributes["file_id"].isin(workflow_files["file_id"])).sum())
checks.append(("workflow_files.repo_id sin repo correspondiente", fk_files_huerfanas))
checks.append(("workflow_locks.file_id sin archivo .md correspondiente", fk_locks_huerfanas))
checks.append(("frontmatter_attributes.file_id sin archivo correspondiente", fk_attrs_huerfanas))

# --- relación 1:1 workflow_files <-> workflow_locks ---
archivos_sin_lock = int((~workflow_files["file_id"].isin(workflow_locks["file_id"])).sum())
locks_por_archivo = workflow_locks.groupby("file_id").size()
archivos_con_mas_de_1_lock = int((locks_por_archivo > 1).sum())
checks.append((".md sin su .lock correspondiente", archivos_sin_lock))
checks.append((".md con más de 1 .lock (rompe 1:1)", archivos_con_mas_de_1_lock))

quality_summary = pd.DataFrame(checks, columns=["comprobación", "casos_encontrados"])
quality_summary

,comprobación,casos_encontrados
0,"Valores ausentes (todas las columnas, las 4 ta...",0
1,Filas duplicadas en repositories,0
2,Filas duplicadas en workflow_files,0
3,Filas duplicadas en workflow_locks,0
4,Filas duplicadas en frontmatter_attributes,0
5,repo_id vacío o duplicado,0
6,file_id vacío o duplicado,0
7,lock_id vacío o duplicado,0
8,attribute_id vacío o duplicado,0
9,workflow_files.repo_id sin repo correspondiente,0


### 3.1 Consistencia interna: ¿el `value` coincide con su `value_type` declarado?

In [5]:
def value_matches_type(value: str, value_type: str) -> bool:
    if value_type == "int":
        try:
            int(value); return True
        except ValueError:
            return False
    if value_type == "float":
        try:
            float(value); return True
        except ValueError:
            return False
    if value_type == "bool":
        return value in ("True", "False")
    if value_type == "null":
        return value == ""
    if value_type == "dict":
        return value == "{}"
    if value_type == "list":
        return value == "[]"
    return True  # "str" acepta cualquier texto

mismatches = frontmatter_attributes[
    ~frontmatter_attributes.apply(lambda r: value_matches_type(r["value"], r["value_type"]), axis=1)
]
print(f"Atributos con value inconsistente respecto a value_type: {len(mismatches)}")
mismatches.head()

Atributos con value inconsistente respecto a value_type: 0


,attribute_id,file_id,key_path,value,value_type


### 3.2 El caso del archivo sin ningún atributo de frontmatter

In [6]:
archivos_sin_atributos = workflow_files[~workflow_files["file_id"].isin(frontmatter_attributes["file_id"])]
print(f"Archivos .md sin ningún atributo de frontmatter: {len(archivos_sin_atributos)} de {len(workflow_files)}")
archivos_sin_atributos[["file_id", "repo_id", "file_name", "file_path", "raw_frontmatter"]]

Archivos .md sin ningún atributo de frontmatter: 1 de 1390


,file_id,repo_id,file_name,file_path,raw_frontmatter
396,397,140,daily-test-improver.md,.github/workflows/daily-test-improver.md,{}


In [7]:
# ¿por qué quedó vacío? -- se revisa si el body contiene un segundo
# delimitador "---" que cierre el bloque de frontmatter
caso = archivos_sin_atributos.iloc[0]
tiene_delimitador_de_cierre = "\n---\n" in caso["body_markdown"][3:]
print("Empieza con '---':", caso["body_markdown"].startswith("---"))
print("Tiene un '---' de cierre más adelante en el archivo:", tiene_delimitador_de_cierre)
print()
print("Primeras líneas del archivo:")
print(caso["body_markdown"][:300])

Empieza con '---': True
Tiene un '---' de cierre más adelante en el archivo: False

Primeras líneas del archivo:
---
  description: |
    A testing-focused repository assistant that runs daily to improve test quality and coverage.
    Can also be triggered on-demand via '/test-assist <instructions>' to perform specific tasks.
    - Discovers and validates build, test, and coverage commands for the repository
 


**Interpretación:** el archivo `daily-test-improver.md` (repo_id 140)
empieza con `---` como si fuera a declarar frontmatter, pero **nunca tiene
un `---` de cierre en todo el archivo** — es decir, el bloque de
frontmatter quedó mal formado en el repositorio de origen (no es un
problema introducido por Miner). El parser de Miner está diseñado
justamente para este caso: si no encuentra un bloque de frontmatter válido,
no lanza una excepción ni descarta el archivo — devuelve `frontmatter={}`
y conserva **todo** el contenido como `body_markdown`, que es exactamente
lo que se observa acá. La fila de `workflow_files` para este archivo es
válida y se conserva (con su `.lock` correspondiente intacto en
`workflow_locks`); simplemente no aporta ninguna fila a
`frontmatter_attributes`, lo cual es coherente y no requiere ningún
tratamiento adicional sobre esa tabla.


## 4. Tratamiento de los problemas encontrados

Resumen de la revisión de calidad (sección 3): **no se encontraron** filas
duplicadas, claves primarias vacías o repetidas, claves foráneas
huérfanas, inconsistencias entre `value` y `value_type`, ni violaciones a
la relación 1:1 entre `workflow_files` y `workflow_locks`. El dataset está
limpio en todos esos aspectos porque la propia CLI de Miner ya impone esas
garantías al construirlo (llaves subrogadas asignadas secuencialmente,
validación de tipos con Pydantic, un par `.md`/`.lock` solo se registra si
ambos archivos se descargaron con éxito, y escritura atómica de las 4
tablas juntas) — no se requiere ningún código de limpieza para esos casos.

El único caso a tratar es el archivo con frontmatter vacío por delimitador
de cierre faltante en el origen (sección 3.2). La decisión adoptada es:

- **No excluir la fila** de `workflow_files`: sigue siendo un archivo de
  workflow GH-AW válido (tiene su `.lock.yml` confirmado, con su fila
  correspondiente en `workflow_locks`), solo que no se pudo extraer su
  frontmatter estructurado. Excluirlo perdería información real sobre la
  existencia del archivo.
- **Agregar una columna derivada `has_frontmatter`** a la tabla procesada,
  para que el Notebook 2 pueda reportar explícitamente la cobertura de
  frontmatter sin tener que re-derivarla, y para que quede documentado que
  la ausencia de atributos en ese archivo es conocida y explicada (no un
  campo opcional simplemente no usado).

Se modifica 1 de 1390 registros (0.07%); no se excluye ningún registro de
ninguna tabla.


In [8]:
workflow_files_processed = workflow_files.copy()
workflow_files_processed["has_frontmatter"] = workflow_files_processed["file_id"].isin(
    frontmatter_attributes["file_id"]
)

n_sin_frontmatter = (~workflow_files_processed["has_frontmatter"]).sum()
print(f"Archivos marcados sin frontmatter estructurado: {n_sin_frontmatter} de {len(workflow_files_processed)}")
workflow_files_processed[["file_id", "file_name", "has_frontmatter"]].head()

Archivos marcados sin frontmatter estructurado: 1 de 1390


,file_id,file_name,has_frontmatter
0,1,backport.md,True
1,2,release-highlights.md,True
2,3,issue-triage.md,True
3,4,daily-test-improver.md,True
4,5,build-failure-analysis.md,True


### Guardar tablas preparadas para el Notebook 2

Se guardan las 4 tablas en `data/processed/`, conservando los datos
originales en `data/raw/` sin modificar. `repositories`, `workflow_locks`
y `frontmatter_attributes` se copian sin cambios (no requirieron
tratamiento); `workflow_files` se guarda con la columna `has_frontmatter`
agregada.


In [9]:
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

repositories.to_parquet(PROCESSED_DIR / "repositories.parquet", index=False)
workflow_files_processed.to_parquet(PROCESSED_DIR / "workflow_files.parquet", index=False)
workflow_locks.to_parquet(PROCESSED_DIR / "workflow_locks.parquet", index=False)
frontmatter_attributes.to_parquet(PROCESSED_DIR / "frontmatter_attributes.parquet", index=False)

print("Tablas guardadas en data/processed/:")
for p in sorted(PROCESSED_DIR.glob("*.parquet")):
    print(" -", p)

Tablas guardadas en data/processed/:
 - data\processed\frontmatter_attributes.parquet
 - data\processed\repositories.parquet
 - data\processed\workflow_files.parquet
 - data\processed\workflow_locks.parquet
